# Software decoder latency versus code distance (simulated, this machine)

**Companion to FT-06 and FT-07 (decoding and real-time decoding).** This measures how long **PyMatching**, running in software on this machine's CPU, takes to decode simulated surface-code syndrome data, as the code distance grows.

**What it is not:**

- It is **not hardware decoder latency**: no FPGA, ASIC or control-system path is involved.
- It is **not streaming**: each shot is a complete block of $d$ rounds decoded at once.
- It is **not a worst-case bound**. These are averages over typical noise at the stated $p$; FT-06 explains why worst-case time matters.

Compare the per-round figure with a 1 µs surface-code cycle only as an order-of-magnitude reference.

In [1]:
import platform, os, time
import numpy as np, stim, pymatching
print('stim', stim.__version__, '| pymatching', pymatching.__version__)
print('machine:', platform.platform(), '|', platform.processor() or platform.machine(), '| logical CPUs:', os.cpu_count())

stim 1.15.0 | pymatching 2.2.2
machine: macOS-27.0.1-arm64-arm-64bit-Mach-O | arm | logical CPUs: 10


In [2]:
def decode_timing(d, p=0.001, shots=2000, seed=7, repeats=3):
    circuit = stim.Circuit.generated(
        'surface_code:rotated_memory_z', distance=d, rounds=d,
        after_clifford_depolarization=p, before_round_data_depolarization=p,
        before_measure_flip_probability=p, after_reset_flip_probability=p)
    matching = pymatching.Matching.from_detector_error_model(circuit.detector_error_model(decompose_errors=True))
    detectors, _ = circuit.compile_detector_sampler(seed=seed).sample(shots, separate_observables=True)
    best = float('inf')
    for _ in range(repeats):  # take the best of a few runs to reduce timer noise
        t0 = time.perf_counter(); matching.decode_batch(detectors); best = min(best, time.perf_counter() - t0)
    per_shot = best / shots
    return circuit.num_detectors, per_shot, per_shot / d

print('d    detectors/shot   µs per shot (d rounds)   µs per round   × a 1 µs cycle')
for d in [3, 5, 7, 9, 11, 13, 15, 17]:
    n, shot, rnd = decode_timing(d)
    print(f'{d:<4} {n:<16} {shot * 1e6:<24.1f} {rnd * 1e6:<14.2f} {rnd * 1e6 / 1.0:.2f}')

d    detectors/shot   µs per shot (d rounds)   µs per round   × a 1 µs cycle
3    24               0.1                      0.02           0.02
5    120              0.4                      0.08           0.08
7    336              1.2                      0.17           0.17
9    720              2.8                      0.31           0.31
11   1320             5.2                      0.47           0.47


13   2184             8.8                      0.68           0.68
15   3360             13.6                     0.91           0.91


17   4896             21.5                     1.27           1.27


## Reading the result

- **Growth with distance.** Time per round grows with distance, because each round has $d^2-1$ checks. On the machine recorded above it rose from about 0.02 µs at $d=3$ to about 1.2 µs at $d=17$, at 0.1% noise. How it grows on typical noise is what Sparse Blossom was engineered for. [Higgott and Gidney](https://arxiv.org/abs/2303.15933) report under 1 µs per round at distance 17 and 0.1% noise on one core, with their own benchmark setup. Differences from that figure here reflect this machine, the noise model, batching and Python overhead.
- **Average versus worst case.** A per-round average below 1 µs does not guarantee real-time operation. The decoder must also keep up during bursts and deliver results within a reaction-time budget (FT-07), and it must do that for every logical qubit at once.
- **What hardware decoders do differently.** They target bounded worst-case latency on streaming data, often with union-find-style algorithms (FT-06, FT-07).